# Python Excel Automation: สร้างรายงานและวิเคราะห์ข้อมูลด้วยสูตร Excel กับ Openpyxl - ตอนที่ 4

**คำอธิบาย:** เรียนรู้การสร้างรายงาน Excel และใช้สูตร Excel (SUMIF, VLOOKUP, TEXT) ด้วย Python openpyxl

**หัวข้อที่ครอบคลุม:**
- สร้าง Worksheet Object สำหรับทำงาน
- ตรวจสอบขนาดคอลัมน์
- ใช้สูตร SUMIF กับเงื่อนไขเฉพาะ
- รวมการอ้างอิงเซลล์ในสูตร
- ใช้สูตร VLOOKUP สำหรับค้นหาข้อมูล
- แปลงวันที่ด้วยฟังก์ชัน TEXT
- ตัวอย่างการใช้งานจริง

In [ ]:
# 📦 ติดตั้ง packages ที่จำเป็น (ข้ามถ้าติดตั้งแล้ว)
try:
    import openpyxl
    print("✅ ติดตั้ง packages ทั้งหมดแล้ว")
except ImportError:
    %pip install openpyxl

✅ Packages already installed.


In [ ]:
# 📚 นำเข้าส่วนประกอบ openpyxl
# load_workbook — เปิดไฟล์ที่มีอยู่
# Workbook — สร้างไฟล์ใหม่
from openpyxl import load_workbook, Workbook

## เตรียมข้อมูล: สร้างข้อมูลตัวอย่าง

In [ ]:
# 📝 สร้างข้อมูล Superstore ตัวอย่าง — มีคอลัมน์สำหรับสาธิต SUMIF, VLOOKUP
# จำลองชุดข้อมูลธุรกิจจริงพร้อมคำสั่งซื้อ ลูกค้า หมวดหมู่
wb = Workbook()
ws = wb.active
ws.title = 'Orders'

headers = ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 
           'Customer Name', 'Segment', 'Country', 'City', 'State', 
           'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 
           'Discount', 'Profit']
ws.append(headers)

data = [
    [1, 'CA-2024-001', '2024-01-15', '2024-01-20', 'Standard', 'John Smith', 'Consumer', 'US', 'LA', 'California', 'Technology', 'Phones', 'iPhone 15', 999, 2, 0, 200],
    [2, 'CA-2024-002', '2024-02-20', '2024-02-25', 'Express', 'Jane Doe', 'Corporate', 'US', 'Houston', 'Texas', 'Furniture', 'Chairs', 'Office Chair', 450, 3, 0.1, 100],
    [3, 'CA-2024-003', '2024-03-10', '2024-03-15', 'Standard', 'Bob Wilson', 'Consumer', 'US', 'NYC', 'New York', 'Office Supplies', 'Paper', 'Copy Paper', 25, 10, 0, 5],
    [4, 'CA-2024-004', '2024-04-05', '2024-04-07', 'Same Day', 'Alice Brown', 'Consumer', 'US', 'SF', 'California', 'Technology', 'Laptops', 'MacBook Pro', 2499, 1, 0, 500],
    [5, 'CA-2024-005', '2024-05-18', '2024-05-22', 'Standard', 'Charlie Lee', 'Corporate', 'US', 'Miami', 'Florida', 'Furniture', 'Tables', 'Conference Table', 1200, 1, 0.15, 200],
    [6, 'CA-2024-006', '2024-06-22', '2024-06-25', 'Express', 'Jane Doe', 'Corporate', 'US', 'Houston', 'Texas', 'Technology', 'Accessories', 'Monitor', 350, 2, 0, 80],
    [7, 'CA-2024-007', '2024-07-30', '2024-08-02', 'Standard', 'John Smith', 'Consumer', 'US', 'LA', 'California', 'Office Supplies', 'Binders', 'Ring Binder', 15, 5, 0, 3],
    [8, 'CA-2024-008', '2024-08-12', '2024-08-14', 'Express', 'Alice Brown', 'Consumer', 'US', 'SF', 'California', 'Technology', 'Phones', 'Samsung S24', 899, 1, 0.1, 150],
]

for row in data:
    ws.append(row)

wb.save('Superstore_Report.xlsx')
print("✅ สร้างข้อมูลตัวอย่างแล้ว")

✅ Sample data created


## 1. โหลด Workbook และตั้งค่า Worksheet Object

In [ ]:
# 📂 โหลด workbook และตั้งค่าอ้างอิง worksheet
# เราจะใช้ตัวแปร 'order' ตลอดทั้ง notebook เพื่ออ้างถึง sheet ข้อมูล
workbook = load_workbook('Superstore_Report.xlsx')
print(f"Sheet ทั้งหมด: {workbook.sheetnames}")

order = workbook['Orders']
print(f"จำนวนแถว: {order.max_row}")
print(f"จำนวนคอลัมน์: {order.max_column}")

Sheets: ['Orders']
Max rows: 9
Max columns: 17


## 2. ตรวจสอบขนาดคอลัมน์และดูหัวตาราง

In [ ]:
# 📋 ดูหัวตารางทั้งหมดและตำแหน่ง
# การรู้ตำแหน่งคอลัมน์สำคัญมากสำหรับการสร้างสูตร
# เลขคอลัมน์ใน openpyxl เริ่มจาก 1 (A=1, B=2, C=3...)
headers_list = [cell.value for cell in order[1]]
print("หัวตารางและตำแหน่ง:")
for i, h in enumerate(headers_list):
    print(f"  คอลัมน์ {i+1}: {h}")

Headers and their positions:
  Column 1: Row ID
  Column 2: Order ID
  Column 3: Order Date
  Column 4: Ship Date
  Column 5: Ship Mode
  Column 6: Customer Name
  Column 7: Segment
  Column 8: Country
  Column 9: City
  Column 10: State
  Column 11: Category
  Column 12: Sub-Category
  Column 13: Product Name
  Column 14: Sales
  Column 15: Quantity
  Column 16: Discount
  Column 17: Profit


In [ ]:
# 🔑 หาค่าที่ไม่ซ้ำในคอลัมน์ State
# เราต้องการรายชื่อรัฐที่ไม่ซ้ำเพื่อสร้างสูตร SUMIF ให้แต่ละรัฐ
# set() รับประกันว่าไม่มีค่าซ้ำ
state_index = headers_list.index('State')
print(f"ตำแหน่งคอลัมน์ 'State': {state_index}")

unique_values = set()
for row in order.iter_rows(min_row=2, min_col=state_index+1, max_col=state_index+1, values_only=True):
    unique_values.add(row[0])
print(f"รัฐที่ไม่ซ้ำ: {unique_values}")

'State' column index: 9
Unique states: {'Florida', 'Texas', 'New York', 'California'}


## 3. ใช้สูตร SUMIF

In [ ]:
# 📊 สูตร SUMIF — รวมค่าที่ตรงเงื่อนไข
# =SUMIF(ช่วงตรวจ, เงื่อนไข, ช่วงรวม)
#   ช่วงตรวจ = คอลัมน์ที่จะตรวจ (State = คอลัมน์ J)
#   เงื่อนไข = ค่าที่ต้องตรง (ชื่อรัฐในเซลล์ A2)
#   ช่วงรวม = คอลัมน์ที่จะรวม (Sales = คอลัมน์ N, Profit = คอลัมน์ Q)
# openpyxl เขียนสูตรเป็นข้อความ — Excel คำนวณเมื่อเปิดไฟล์
report_ws = workbook.create_sheet('Report')

report_ws['A1'] = 'State'
report_ws['B1'] = 'Total Sales'
report_ws['C1'] = 'Total Profit'

for i, state in enumerate(sorted(unique_values), start=2):
    report_ws.cell(row=i, column=1, value=state)
    report_ws.cell(row=i, column=2, value=f'=SUMIF(Orders!J:J,A{i},Orders!N:N)')
    report_ws.cell(row=i, column=3, value=f'=SUMIF(Orders!J:J,A{i},Orders!Q:Q)')

print("✅ เพิ่มสูตร SUMIF แล้ว")
for row in report_ws.iter_rows(min_row=1, values_only=True):
    print(row)

✅ SUMIF formulas added
('State', 'Total Sales', 'Total Profit')
('California', '=SUMIF(Orders!J:J,A2,Orders!N:N)', '=SUMIF(Orders!J:J,A2,Orders!Q:Q)')
('Florida', '=SUMIF(Orders!J:J,A3,Orders!N:N)', '=SUMIF(Orders!J:J,A3,Orders!Q:Q)')
('New York', '=SUMIF(Orders!J:J,A4,Orders!N:N)', '=SUMIF(Orders!J:J,A4,Orders!Q:Q)')
('Texas', '=SUMIF(Orders!J:J,A5,Orders!N:N)', '=SUMIF(Orders!J:J,A5,Orders!Q:Q)')


## 4. เพิ่มสูตร VLOOKUP

In [ ]:
# 🔎 สูตร VLOOKUP — ค้นหาค่าแล้วส่งกลับข้อมูลจากคอลัมน์อื่น
# =VLOOKUP(ค่าค้นหา, ตารางข้อมูล, ลำดับคอลัมน์, ตรงทั้งหมด)
#   ค่าค้นหา = ชื่อลูกค้า (A2)
#   ตารางข้อมูล = ช่วงที่จะค้น (Orders!F:N = คอลัมน์ F ถึง N)
#   ลำดับคอลัมน์ = คอลัมน์ที่จะส่งกลับ (6 = Category, 9 = Sales)
#   FALSE = ต้องตรงทั้งหมดเท่านั้น
lookup_ws = workbook.create_sheet('Lookup')

lookup_ws['A1'] = 'Customer Name'
lookup_ws['B1'] = 'Category (VLOOKUP)'
lookup_ws['C1'] = 'Sales (VLOOKUP)'

customers = ['John Smith', 'Jane Doe', 'Alice Brown', 'Bob Wilson', 'Charlie Lee']

for i, customer in enumerate(customers, start=2):
    lookup_ws.cell(row=i, column=1, value=customer)
    lookup_ws.cell(row=i, column=2, value=f'=VLOOKUP(A{i},Orders!F:N,6,FALSE)')
    lookup_ws.cell(row=i, column=3, value=f'=VLOOKUP(A{i},Orders!F:N,9,FALSE)')

print("✅ เพิ่มสูตร VLOOKUP แล้ว")
for row in lookup_ws.iter_rows(min_row=1, values_only=True):
    print(row)

✅ VLOOKUP formulas added
('Customer Name', 'Category (VLOOKUP)', 'Sales (VLOOKUP)')
('John Smith', '=VLOOKUP(A2,Orders!F:N,6,FALSE)', '=VLOOKUP(A2,Orders!F:N,9,FALSE)')
('Jane Doe', '=VLOOKUP(A3,Orders!F:N,6,FALSE)', '=VLOOKUP(A3,Orders!F:N,9,FALSE)')
('Alice Brown', '=VLOOKUP(A4,Orders!F:N,6,FALSE)', '=VLOOKUP(A4,Orders!F:N,9,FALSE)')
('Bob Wilson', '=VLOOKUP(A5,Orders!F:N,6,FALSE)', '=VLOOKUP(A5,Orders!F:N,9,FALSE)')
('Charlie Lee', '=VLOOKUP(A6,Orders!F:N,6,FALSE)', '=VLOOKUP(A6,Orders!F:N,9,FALSE)')


## 5. แปลงวันที่ด้วยฟังก์ชัน TEXT

In [ ]:
# 📅 ฟังก์ชัน TEXT — จัดรูปแบบวันที่เป็นข้อความที่อ่านง่ายใน Excel
# =TEXT(เซลล์, "รูปแบบ") แปลงเซลล์วันที่เป็นข้อความตามรูปแบบ
# "MMM DD, YYYY" → "Jan 15, 2024"
# เราเพิ่มคอลัมน์ใหม่ใน sheet Orders
new_col = order.max_column + 1
order.cell(row=1, column=new_col, value='Formatted Date')

for row_num in range(2, order.max_row + 1):
    order.cell(row=row_num, column=new_col, 
               value=f'=TEXT(C{row_num},"MMM DD, YYYY")')

print(f"✅ เพิ่มสูตร TEXT ในคอลัมน์ที่ {new_col}")

✅ TEXT formula added in column 18


## 6. เพิ่มสูตรเพิ่มเติม: COUNTIF, AVERAGEIF

In [ ]:
# 📊 COUNTIF และ AVERAGEIF — สูตร Excel ที่มีประโยชน์เพิ่มเติม
# COUNTIF = นับจำนวนแถวที่ตรงเงื่อนไข
# AVERAGEIF = คำนวณค่าเฉลี่ยของค่าที่ตรงเงื่อนไข
# รูปแบบเดียวกับ SUMIF: (ช่วงตรวจ, เงื่อนไข[, ช่วงเฉลี่ย])
report_ws['D1'] = 'Order Count'
report_ws['E1'] = 'Avg Sales'

for i in range(2, 2 + len(unique_values)):
    report_ws.cell(row=i, column=4, value=f'=COUNTIF(Orders!J:J,A{i})')
    report_ws.cell(row=i, column=5, value=f'=AVERAGEIF(Orders!J:J,A{i},Orders!N:N)')

print("✅ เพิ่มสูตร COUNTIF และ AVERAGEIF แล้ว")

✅ COUNTIF and AVERAGEIF formulas added


## 7. บันทึกรายงาน

In [ ]:
# 💾 บันทึกและปิด
# ⚠️ สูตรถูกบันทึกเป็นข้อความ — เปิดใน Excel เพื่อดูผลลัพธ์ที่คำนวณแล้ว
workbook.save('openpyxl_reports_formulas.xlsx')
print(f"✅ บันทึกรายงานแล้ว พร้อม sheet: {workbook.sheetnames}")
print("\n💡 เปิดไฟล์ใน Excel เพื่อดูผลลัพธ์ของสูตรที่คำนวณแล้ว")
workbook.close()

✅ Report saved with sheets: ['Orders', 'Report', 'Lookup']

💡 Open the file in Excel to see calculated formula results.


---

## 🎮 Mini Project: สร้างรายงานยอดขายอัตโนมัติ

ทดสอบการใส่สูตร Excel ด้วย openpyxl!

> 💡 **คำตอบ:** ดูไฟล์ `answers/05_answers.ipynb`

---

### โจทย์ที่ 1: สร้าง Sheet สรุปยอดขายตาม Region 🟢
ใช้ไฟล์ `Random Sales Data.xlsx`:
1. สร้าง sheet ชื่อ `"Region_Summary"`
2. ใส่ header: `Region`, `Total Sales`, `Order Count`
3. หา unique Region จากข้อมูล
4. ใส่สูตร `SUMIF` ใน Total Sales (รวมยอดขายตาม Region)
5. ใส่สูตร `COUNTIF` ใน Order Count (นับจำนวนออเดอร์ตาม Region)
6. บันทึกไฟล์

> 💡 Hint: สูตร `=SUMIF(Orders!J:J,"East",Orders!N:N)`

In [ ]:
# โจทย์ที่ 1: สร้าง Sheet สรุปยอดขายตาม Region
# เขียนโค้ดของคุณที่นี่ 👇



### โจทย์ที่ 2: เพิ่มสูตร AVERAGEIF และ VLOOKUP 🟡
ต่อจากโจทย์ที่ 1:
1. เพิ่มคอลัมน์ `Avg Sale` ในsheet `"Region_Summary"` — ใส่สูตร `AVERAGEIF`
2. สร้าง sheet ใหม่ `"Lookup"` ที่มี:
   - คอลัมน์ A: ชื่อ Sales Representative (พิมพ์ 3 ชื่อจากข้อมูลจริง)
   - คอลัมน์ B: ใส่สูตร `VLOOKUP` ที่ดึง Region ของ Sales Rep นั้น
3. บันทึกไฟล์

> 💡 Hint: `=AVERAGEIF(range, criteria, avg_range)` / `=VLOOKUP(value, table, col, FALSE)`

In [ ]:
# โจทย์ที่ 2: AVERAGEIF + VLOOKUP
# เขียนโค้ดของคุณที่นี่ 👇

